# Stage 3 DPD Safe-Zone — Phase B: notebook setup

**ПРОСТЫМ ЯЗЫКОМ:** Загружаем 4 сырых CSV из SQL-выгрузки (`stage3_safezone_rolling_extract.sql`), строим для каждого из 12 отчётных месяцев 6-месячное окно DPD по займу, и определяем состояние реструктуризации по каждому месяцу окна — **три состояния, не «да/нет»**: `active` (каникулы действовали), `not_active` (точно не действовали), `unknown` (событие есть, но дат каникул в источнике нет — судить не можем). Отсюда `restr_active_pct` и `restr_unknown_pct` по каждому займу. Это подготовка данных для Фазы C (симуляция порога) — сама симуляция здесь ещё не реализована.

See [`docs/analysis/stage3_safezone_plan.md`](../docs/analysis/stage3_safezone_plan.md) for the full plan (Phases A–E).

**Before running:** point `RAW_DATA_DIR` below at your exported CSVs and confirm the filenames in `FILES` match what you actually exported — the diagnostic cell right after lists what's actually in the folder.

**What to check before moving to Phase C:** the row-count sanity check, then `mean_unknown_pct` and `pct_event_dated`. If the unknown share is material, the restructured-vs-not split in Phase C rests on a denominator of unknown quality and has to be reported that way — not quietly folded into "not restructured."

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DATA_DIR = Path(r"C:\project_mz\surau\DPDRelaxing\raw_data")

# Adjust these to your actual exported filenames if they differ.
FILES = {
    "report_dates": "report_dates.csv",
    "stage3_pool": "stage3_pool.csv",
    "dpd_panel": "dpd_panel.csv",
    "restructuring_events": "restructuring_events.csv",
}

LAST_ASOF = pd.Timestamp("2026-07-01")
LOOKBACK_MONTHS = 6

In [ ]:
print("CSV files found in RAW_DATA_DIR:")
for f in sorted(RAW_DATA_DIR.glob("*.csv")):
    print(" ", f.name)

## Restore column headers

**ПРОСТЫМ ЯЗЫКОМ:** SSMS-экспорт в CSV теряет заголовок колонок, но порядок колонок сохраняется таким, как в SELECT в `stage3_safezone_rolling_extract.sql`. Ниже — четыре списка имён в точном порядке запроса (для `stage3_pool` — все 69 колонок `CL_PORTFOLIO_2` в табличном порядке, плюс `portfolio_label`/`portfolio_asof`) и функция-загрузчик, которая присваивает имена, проверяет число колонок и на всякий случай снимает заголовок, если он всё-таки просочился.

If the SQL result set changes, update the matching `COLUMNS_*` list here — the count check below will fail loudly rather than silently misaligning columns.

In [ ]:
import csv

# stage3_safezone_rolling_extract.sql §0 -- SELECT * FROM ##SAFEZONE_REPORT_DATES
COLUMNS_REPORT_DATES = ["asof_date", "months_back", "portfolio_label"]

# §1 -- SELECT a.*, rd.portfolio_label, rd.asof_date AS portfolio_asof FROM CL_PORTFOLIO_2 a ...
# a.* is every CL_PORTFOLIO_2 column in table order (confirmed via INFORMATION_SCHEMA.COLUMNS).
COLUMNS_STAGE3_POOL = [
    "contract_number", "status", "granting_date", "first_pmt_date", "outstanding",
    "outstanding_overdue", "interest", "overdue_interest",
    "accrued_interest_on_overdue_outstanding", "overdue_interest_on_overdue_outstanding",
    "penalties", "termination_commissions", "loan_servicing_commissions",
    "overdue_commissions_income", "overdue_days_principal", "overdue_days_interest",
    "dpd", "category", "percent", "ifrs", "provisions_calculated", "loan_purpose",
    "subproduct", "discount_1434", "discount_1435", "discount_1773", "fees_1860",
    "attribute_marker", "date_of_attribute_marker", "filial", "merchant_city",
    "LoanDuration", "FKLogin", "IIN", "CARBRAND", "teh_overdfraft",
    "overdue_commissions", "overdue_penalties", "od", "balance", "product", "basket",
    "90+", "90+sum", "fin_date_short", "date", "tag", "tag_1", "rezident", "valuta",
    "fiziki/yuriki", "IFRS1877", "IFRS1845", "maxDPD", "ifrs18770_DEB",
    "ifrs18771_WTRAF_i_PENII", "OD_percent", "tarif", "discount_1774", "discount_1775",
    "discount_1784", "balance_with_discount", "provisions_total", "DISCOUNT_1484",
    "DISCOUNT_14341", "DISCOUNT_14342", "DISCOUNT_1485", "DISCOUNT_17731",
    "DISCOUNT_17732",
    # appended by the extract query, not part of CL_PORTFOLIO_2 itself:
    "portfolio_label", "portfolio_asof",
]

# §2 -- SELECT p.contract_number, p.[date] AS snap_date, p.[dpd], p.[category], p.[balance],
#         p.[balance_with_discount], p.[provisions_total], p.[tag]
COLUMNS_DPD_PANEL = [
    "contract_number", "snap_date", "dpd", "category", "balance",
    "balance_with_discount", "provisions_total", "tag",
]

# §3 -- SELECT r.* FROM [Dictionaries].[risk_analytics].[restructuring_v2] r ...
COLUMNS_RESTRUCTURING_EVENTS = [
    "dlcr_gid", "dlcr$source", "loan_id", "restructuring_date", "new_interest_rate",
    "days_past_due_at_restructuring", "new_maturity_date", "financial_deterioration_flag",
    "payment_deferral", "canc_date", "grace_od_begin_date", "grace_int_begin_date",
    "grace_od_end_date", "grace_int_end_date", "report_date",
]


def read_headerless_csv(
    path: Path, columns: list, parse_dates: list = None
) -> pd.DataFrame:
    """Read a CSV whose header row was dropped on export, restoring names from the SQL
    SELECT order. Also tolerates a header row that slipped through anyway (detected by
    comparing the first row to `columns` and skipped), and fails loudly on a column-count
    mismatch instead of silently misaligning data under the wrong names."""
    with open(path, newline="", encoding="utf-8-sig") as f:
        first_row = next(csv.reader(f))
    skip = 1 if [v.strip() for v in first_row] == columns else 0

    df = pd.read_csv(path, header=None, skiprows=skip, names=columns)
    if len(df.columns) != len(columns):
        raise ValueError(
            f"{path.name}: expected {len(columns)} columns (SQL SELECT order), found "
            f"{len(df.columns)} in the file -- update the matching COLUMNS_* list to match "
            "stage3_safezone_rolling_extract.sql."
        )
    if parse_dates:
        for col in parse_dates:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

## Load raw extracts

Matches the four result sets from `sql/stage3_safezone_rolling_extract.sql` §0/§1/§2/§3, with headers restored by `read_headerless_csv`.

In [ ]:
report_dates = read_headerless_csv(
    RAW_DATA_DIR / FILES["report_dates"], COLUMNS_REPORT_DATES, parse_dates=["asof_date"]
)

stage3_pool = read_headerless_csv(
    RAW_DATA_DIR / FILES["stage3_pool"],
    COLUMNS_STAGE3_POOL,
    parse_dates=["date", "portfolio_asof"],
)

dpd_panel = read_headerless_csv(
    RAW_DATA_DIR / FILES["dpd_panel"], COLUMNS_DPD_PANEL, parse_dates=["snap_date"]
)

restructuring_events = read_headerless_csv(
    RAW_DATA_DIR / FILES["restructuring_events"],
    COLUMNS_RESTRUCTURING_EVENTS,
    parse_dates=[
        "restructuring_date",
        "new_maturity_date",
        "canc_date",
        "grace_od_begin_date",
        "grace_od_end_date",
        "grace_int_begin_date",
        "grace_int_end_date",
        "report_date",
    ],
)

## Sanity check — row counts against what SQL Server reported (23.07.2026)

In [ ]:
expected = {
    "report_dates": 12,
    "stage3_pool": 481_818,
    "dpd_panel": 1_080_891,
    "restructuring_events": 91_679,
}
actual = {
    "report_dates": len(report_dates),
    "stage3_pool": len(stage3_pool),
    "dpd_panel": len(dpd_panel),
    "restructuring_events": len(restructuring_events),
}
for name, expected_count in expected.items():
    got = actual[name]
    flag = "OK" if got == expected_count else "CHECK — differs from the SQL-side count"
    print(f"{name:22s} expected {expected_count:>10,}  got {got:>10,}  [{flag}]")

## Per-portfolio 6-month lookback window

For a given `portfolio_asof`, slice `dpd_panel` down to the `LOOKBACK_MONTHS` ending at that date (inclusive), and label each row with a `month_offset` (0 = the portfolio's own month, negative = further back).

In [ ]:
def build_lookback_dpd(
    dpd_panel: pd.DataFrame, portfolio_asof: pd.Timestamp, lookback_months: int = LOOKBACK_MONTHS
) -> pd.DataFrame:
    """DPD/category rows for the lookback window ending at portfolio_asof (inclusive)."""
    window_start = portfolio_asof - pd.DateOffset(months=lookback_months - 1)
    window = dpd_panel[
        (dpd_panel["snap_date"] >= window_start) & (dpd_panel["snap_date"] <= portfolio_asof)
    ].copy()
    window["month_offset"] = (
        (window["snap_date"].dt.year - portfolio_asof.year) * 12
        + (window["snap_date"].dt.month - portfolio_asof.month)
    )
    return window

## Restructuring state per month — `active` / `not_active` / `unknown`

**ПРОСТЫМ ЯЗЫКОМ:** три состояния вместо «да/нет». Займ считается **под реструктуризацией** (`active`), если на дату среза есть уже начавшееся и не отменённое событие, и срез попадает внутрь окна каникул — по основному долгу (`grace_od_*`) или по вознаграждению (`grace_int_*`). Если событие есть, но **даты каникул в источнике не заполнены** — это `unknown`: реструктуризация была, покрывала ли она этот месяц — мы не знаем. `not_active` ставим только когда действительно можем это утверждать.

Why three states and not a boolean: an event with **no grace dates populated at all** cannot be judged either way. The previous boolean folded that case into `False`, which reports a genuinely restructured loan as "no payment holiday" — and quietly biases every restructured-vs-not comparison in Phase C, on a denominator of unknown quality.

| State | Meaning |
|---|---|
| `active` | a qualifying event's grace window covers this snapshot |
| `unknown` | a qualifying event exists but carries no usable grace dates to check against |
| `not_active` | no qualifying event at all, **or** every qualifying event has dates and this snapshot falls outside all of them |

"Qualifying" = `restructuring_date <= snap_date` and not cancelled by then (`canc_date` null or later). A loan can match several events — states aggregate with `any()` across all of them, since an older event's grace window can still be running. One *undated* qualifying event is enough to make the month `unknown`, even if a sibling event has dates that don't cover it: we cannot rule coverage out.

Read `restr_unknown_pct` as a **data-quality** reading, not a risk reading — a high value means the source can't answer for that loan, not that the loan is safe.

In [ ]:
GRACE_PAIRS = [
    ("grace_od_begin_date", "grace_od_end_date"),      # principal holiday
    ("grace_int_begin_date", "grace_int_end_date"),    # interest holiday
]


def _has_grace_dates(events: pd.DataFrame) -> pd.Series:
    """Per event row: is at least one grace pair fully populated (both ends)?
    A half-populated pair is unusable — an open-ended window can't be tested."""
    present = pd.Series(False, index=events.index)
    for begin_col, end_col in GRACE_PAIRS:
        present = present | (events[begin_col].notna() & events[end_col].notna())
    return present


def classify_restructuring(
    dpd_window: pd.DataFrame, restructuring_events: pd.DataFrame
) -> pd.DataFrame:
    """One row per (contract_number, snap_date) with restr_state in
    {'active', 'not_active', 'unknown'} — see the markdown above for the rules."""
    events = restructuring_events.rename(columns={"loan_id": "contract_number"})
    merged = dpd_window.merge(events, on="contract_number", how="left")

    # Started by this snapshot and not cancelled before it. NaT comparisons are False,
    # so contracts with no event at all fall out here and end up 'not_active'.
    qualifies = (merged["restructuring_date"] <= merged["snap_date"]) & (
        merged["canc_date"].isna() | (merged["canc_date"] > merged["snap_date"])
    )

    dated = _has_grace_dates(merged)
    in_grace = pd.Series(False, index=merged.index)
    for begin_col, end_col in GRACE_PAIRS:
        in_grace = in_grace | (
            merged[begin_col].notna()
            & merged[end_col].notna()
            & (merged["snap_date"] >= merged[begin_col])
            & (merged["snap_date"] <= merged[end_col])
        )

    merged["_active"] = qualifies & in_grace
    merged["_undated"] = qualifies & ~dated

    per_month = (
        merged.groupby(["contract_number", "snap_date"])[["_active", "_undated"]]
        .any()
        .reset_index()
    )
    # Priority: a confirmed covering window wins; otherwise any undated qualifying
    # event makes the month unanswerable; only then can we assert not_active.
    per_month["restr_state"] = np.where(
        per_month["_active"],
        "active",
        np.where(per_month["_undated"], "unknown", "not_active"),
    )
    return per_month[["contract_number", "snap_date", "restr_state"]]


def compute_restr_state_pct(
    dpd_window: pd.DataFrame, restructuring_events: pd.DataFrame
) -> pd.DataFrame:
    """Share of each contract's OBSERVED months in the window per state.
    Columns: restr_active_pct / restr_unknown_pct / restr_not_active_pct (sum to 1.0).

    Denominator is months actually present in the panel, not LOOKBACK_MONTHS — a loan
    with a missing snapshot must not be diluted toward zero as if that month were clean.
    """
    states = classify_restructuring(dpd_window, restructuring_events)
    counts = (
        states.pivot_table(
            index="contract_number",
            columns="restr_state",
            values="snap_date",
            aggfunc="count",
            fill_value=0,
        )
        .reindex(columns=["active", "unknown", "not_active"], fill_value=0)
    )
    pct = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)
    pct.columns = ["restr_active_pct", "restr_unknown_pct", "restr_not_active_pct"]
    return pct.reset_index()

## Build the per-month state shares for all 12 portfolio months

Two matrices (loan × portfolio month): `restr_active_pct` — the risk signal Phase C splits on — and `restr_unknown_pct` — how much of that split is guesswork. Look at the summary underneath before trusting either: a month with a high mean unknown share cannot support a "restructured vs not" comparison at all.

In [ ]:
restr_state_pct_by_portfolio = {}

for _, row in report_dates.iterrows():
    window = build_lookback_dpd(dpd_panel, row["asof_date"])
    restr_state_pct_by_portfolio[row["portfolio_label"]] = compute_restr_state_pct(
        window, restructuring_events
    ).set_index("contract_number")

restr_active_pct = pd.concat(
    {k: v["restr_active_pct"] for k, v in restr_state_pct_by_portfolio.items()}, axis=1
)
restr_unknown_pct = pd.concat(
    {k: v["restr_unknown_pct"] for k, v in restr_state_pct_by_portfolio.items()}, axis=1
)

# Per-month readout. mean_unknown_pct is the one to watch: it caps how much weight the
# restructured-vs-not split in Phase C can carry for that month.
pd.DataFrame(
    {
        "mean_active_pct": restr_active_pct.mean().round(3),
        "mean_unknown_pct": restr_unknown_pct.mean().round(3),
        "loans_with_any_unknown_month": (restr_unknown_pct > 0).sum(),
        "loans_in_window": restr_active_pct.notna().sum(),
    }
)

## Transparency metric — coverage *and* usability of the restructuring source

**ПРОСТЫМ ЯЗЫКОМ:** два разных вопроса, и оба нужны. Первый — у какой доли пула вообще есть событие реструктуризации. Второй — у какой доли этих событий заполнены даты каникул, без которых проверка на «активна ли она в этом месяце» не работает. Высокий `pct_with_event` при низком `pct_event_dated` означает: реструктуризации были, но когда именно они действовали — источник не знает.

Phase B of the plan asks for "% of the population with a restructuring event **defined vs. not**". `pct_with_event` alone answers the weaker question — a loan can have an event on record and still be unanswerable. Both columns together are the actual transparency metric, and `pct_event_dated` is what bounds the credibility of the Phase C split.

In [ ]:
def restructuring_coverage_summary(
    stage3_pool: pd.DataFrame, restructuring_events: pd.DataFrame
) -> pd.DataFrame:
    """Per portfolio month: how many Stage 3 loans have a restructuring event at all,
    and how many have one with usable grace dates.

    pct_event_dated is deliberately a share OF LOANS WITH AN EVENT, not of the whole
    pool — it answers "when we do have an event, can we use it?", which is the question
    that governs whether restr_active_pct means anything for that month.
    """
    events = restructuring_events.assign(
        has_grace_dates=_has_grace_dates(restructuring_events)
    )
    with_event = set(events["loan_id"].unique())
    with_dated_event = set(events.loc[events["has_grace_dates"], "loan_id"].unique())

    tagged = stage3_pool.assign(
        has_restructuring_event=lambda d: d["contract_number"].isin(with_event),
        has_dated_event=lambda d: d["contract_number"].isin(with_dated_event),
    )
    summary = tagged.groupby("portfolio_label").agg(
        total_loans=("contract_number", "count"),
        with_event=("has_restructuring_event", "sum"),
        with_dated_event=("has_dated_event", "sum"),
    )
    summary["pct_with_event"] = (
        summary["with_event"] / summary["total_loans"] * 100
    ).round(1)
    summary["pct_event_dated"] = (
        summary["with_dated_event"] / summary["with_event"].replace(0, np.nan) * 100
    ).round(1)
    return summary.sort_index()


event_fill = _has_grace_dates(restructuring_events)
print(
    f"Event-level grace-date fill rate: {event_fill.mean() * 100:.1f}% "
    f"({int(event_fill.sum()):,} of {len(event_fill):,} event rows carry a usable pair)"
)

coverage = restructuring_coverage_summary(stage3_pool, restructuring_events)
coverage

## Censoring events — загрузка и аудит покрытия

**ПРОСТЫМ ЯЗЫКОМ:** займ может исчезнуть из портфеля не потому, что вылечился, а потому что его продали, списали или простили. Такие выходы надо цензурировать — иначе они зачтутся как «безопасно вылечился», и любой порог будет выглядеть безопаснее, чем он есть. Здесь грузим `censoring_events.csv` (из `scripts/writeoff_restoration_scan.py`) и проверяем, **за какие месяцы у нас вообще есть чем цензурировать**.

**Ключевое различие: «источника нет» ≠ «событий не было».** Первое оставляет re-default rate нижней границей, второе делает его точным. Разница между этими двумя утверждениями — это разница между «мы не знаем» и «мы знаем, что ничего не было», и в отчёте регулятору на неё смотрят в первую очередь. Поэтому подтверждения записаны в коде явно, с указанием источника и даты, а не растворены в допущениях.

| Тип выхода | Статус |
|---|---|
| Продажи | Источник — `Prodaja&Proschenie_12_2025` (SQL), только 12.2025. **По остальным месяцам подтверждено: продаж не было** (БРМ, 27.07.2026) → покрытие полное, а не слепое пятно. |
| Списания / прощения | Источник — «Приложение №1 (Credilogic)»: 08, 10, 12 · 2025 и 04, 05, 06 · 2026. **По остальным шести месяцам подтверждено: списаний не было** (БРМ, 27.07.2026) → покрытие полное. |

Масштаб, ради которого это различие важно: в декабре, единственном месяце с обоими источниками, **21 341 продажа против 5 016 списаний**. Если бы продажи в других месяцах были и мы их не видели, невидимой оказалась бы бо́льшая половина выходов. Подтверждение БРМ снимает именно этот риск.

Отчёт ниже строится **по списку месяцев панели, а не по содержимому CSV** — месяц, которого нет в файле, обязан появиться строкой со своим статусом, а не исчезнуть.

**Обе оговорки сняты 27.07.2026.** Оба типа выхода теперь по каждому из двенадцати месяцев либо подтверждены событиями, либо подтверждены как не происходившие, — и оговорка про нижнюю границу уходит из Фазы C совсем: re-default rate становится точным числом, а не оценкой снизу. Обратите внимание, что это утверждение держится на двух устных подтверждениях, а не на данных: если хотя бы одно из них будет отозвано, соответствующие месяцы обязаны вернуться в `НЕ ПОДТВЕРЖДЕНО`, а не остаться «как считали раньше». Проверка ниже (`declared_but_present`) ловит обратный случай — когда архив при следующем прогоне всё-таки отдаст события за месяц, объявленный пустым.

In [ ]:
CENSORING_FILE = "censoring_events.csv"

WRITEOFF_SOURCE = "Приложение №1 (Credilogic)"
SALES_SOURCE = {pd.Period("2025-12", freq="M"): "Prodaja&Proschenie_12_2025 (SQL)"}

# DECLARED FACT, not an inference from a missing file: sales took place in 12.2025 only
# (БРМ, 27.07.2026). Recorded explicitly and dated because "no sale register exists" and
# "no sales happened" lead to opposite conclusions — the first leaves the re-default rate
# a lower bound, the second makes it exact. Only the second is true here, and an auditor
# is entitled to see which one we relied on.
SALES_DID_NOT_OCCUR = "продаж не было (БРМ, 27.07.2026)"

# Months where it is likewise confirmed that NO write-off/forgiveness batch ran
# (БРМ, 27.07.2026). Same standard as the sales declaration above and for the same
# reason: these six months are absent from the Credilogic archive, and absence alone
# establishes only that we have no file — not that nothing happened. With the
# confirmation they are covered; without it their re-default rate would stay a lower
# bound. Listed literally rather than derived as "whatever the archive lacks", so that
# a future panel month cannot inherit a confirmation nobody gave for it.
WRITEOFF_DID_NOT_OCCUR: set = {
    pd.Period("2025-09", freq="M"),
    pd.Period("2025-11", freq="M"),
    pd.Period("2026-01", freq="M"),
    pd.Period("2026-02", freq="M"),
    pd.Period("2026-03", freq="M"),
    pd.Period("2026-07", freq="M"),
}
WRITEOFF_DID_NOT_OCCUR_SOURCE = "списаний не было (БРМ, 27.07.2026)"

censoring_events = pd.read_csv(
    RAW_DATA_DIR / CENSORING_FILE, parse_dates=["event_date"], encoding="utf-8-sig"
)
censoring_events["event_month"] = censoring_events["event_date"].dt.to_period("M")


def censoring_coverage_report(
    censoring_events: pd.DataFrame, report_dates: pd.DataFrame
) -> pd.DataFrame:
    """One row per PANEL month: for each exit type, do we have events, a confirmation
    that none occurred, or neither?

    Driven by the panel's month list, never by the CSV's contents — a month absent from
    the file has to surface as a row, not vanish. The distinction the table exists to
    keep visible: an empty month is only fully covered when someone has *confirmed* it
    was event-free. Otherwise it is unconfirmed, and its re-default rate is a lower
    bound rather than a number.
    """
    panel = pd.PeriodIndex(
        pd.to_datetime(report_dates["asof_date"]).dt.to_period("M").unique(), freq="M"
    ).sort_values()
    found = (
        censoring_events.groupby("event_month")
        .agg(
            writeoff_rows=("contract_number", "size"),
            writeoff_contracts=("contract_number", "nunique"),
        )
        .rename_axis("panel_month")
    )

    rep = pd.DataFrame(index=panel.rename("panel_month")).join(found)
    rep[["writeoff_rows", "writeoff_contracts"]] = (
        rep[["writeoff_rows", "writeoff_contracts"]].fillna(0).astype(int)
    )
    has_writeoffs = rep["writeoff_contracts"] > 0
    declared_no_writeoffs = rep.index.isin(WRITEOFF_DID_NOT_OCCUR)

    rep["sales"] = [SALES_SOURCE.get(m, SALES_DID_NOT_OCCUR) for m in rep.index]
    rep["writeoffs"] = np.select(
        [has_writeoffs, declared_no_writeoffs],
        [WRITEOFF_SOURCE, WRITEOFF_DID_NOT_OCCUR_SOURCE],
        default="НЕ ПОДТВЕРЖДЕНО",
    )
    rep["coverage"] = np.where(
        has_writeoffs | declared_no_writeoffs, "полное", "НЕПОЛНОЕ"
    )
    return rep


censoring_coverage = censoring_coverage_report(censoring_events, report_dates)
print(censoring_coverage.to_string())

unconfirmed = censoring_coverage.index[censoring_coverage["coverage"] == "НЕПОЛНОЕ"]

# A declaration can go stale in one direction only: someone says a month was empty, and
# a later scanner run finds events in it. Then the declaration is wrong, not the archive
# — and since the coverage table gives has_writeoffs priority, the month would silently
# read "полное" off the events while the stale confirmation sat unnoticed in the code.
declared_but_present = sorted(
    m for m in WRITEOFF_DID_NOT_OCCUR
    if m in censoring_coverage.index and censoring_coverage.loc[m, "writeoff_contracts"] > 0
)
if declared_but_present:
    raise AssertionError(
        "Подтверждение «списаний не было» противоречит архиву по месяцам: "
        + ", ".join(str(m) for m in declared_but_present)
        + ". Либо подтверждение относилось к другому периоду, либо архив пополнили — "
        "разберитесь до того, как считать re-default: сейчас цензурирование по этим "
        "месяцам построено на взаимоисключающих утверждениях."
    )

stray = sorted(m for m in WRITEOFF_DID_NOT_OCCUR if m not in censoring_coverage.index)
if stray:
    print("[COVERAGE] в WRITEOFF_DID_NOT_OCCUR есть месяцы вне панели (ни на что не "
          "влияют, но список стоит почистить): " + ", ".join(str(m) for m in stray))

print(f"\n[COVERAGE] {len(censoring_events):,} censoring rows, "
      f"{censoring_events['contract_number'].nunique():,} distinct contracts")
print(f"[COVERAGE] sales: 12.2025 only, confirmed — every other month is covered by the "
      f"declaration, not blind")
if len(unconfirmed):
    print(f"[COVERAGE] {len(unconfirmed)} of {len(censoring_coverage)} months have NO "
          f"write-off events and NO confirmation that none occurred:")
    print(f"           {', '.join(str(m) for m in unconfirmed)}")
    print("           Until confirmed, an exit in those months is indistinguishable from\n"
          "           a cure and their re-default rate is a LOWER BOUND. Add them to\n"
          "           WRITEOFF_DID_NOT_OCCUR once checked.")
else:
    print("[COVERAGE] every panel month is either evidenced or confirmed event-free — "
          "censoring is complete, no lower-bound caveat needed")

### С какого месяца займ считается цензурированным

**Событие месяца M цензурирует займ начиная с M+1. Сам месяц M остаётся наблюдаемым.**

Не «с M» — и это не придирка. Декабрь даёт проверяемый факт: списание на внебаланс прошло **30.12.2025**, при этом займы ещё присутствуют в срезе 01.12.2025 и исчезают только к 01.01.2026. Если цензурировать «с 2025-12-01», мы выбросим месяц реального наблюдения у каждого цензурированного займа — систематически, в одну сторону, на 22 тысячах займов.

Точность дат в архиве — только до месяца (`date_precision='month'` по всем строкам: в именах файлов «Приложения №1» дат нет), поэтому граница M+1 — единственная защитимая: чтобы сделать точнее, нужен день, а его нет.

Если один контракт встречается в нескольких месяцах, берётся **первое** событие. Оговорка: архив называется «списание-**восстановление**», и если в «Приложение №1» попадают ещё и восстановления, то часть таких повторов — вернувшиеся в портфель займы, которые цензурировать нельзя. Пока в файле читается только колонка с номером контракта, тип операции не виден — открытый вопрос, помечен в плане.

In [ ]:
def censored_from_month(censoring_events: pd.DataFrame) -> pd.Series:
    """Per contract: the first month from which it must be treated as censored (= M+1
    of its earliest event). Index = contract_number."""
    first_event = censoring_events.groupby("contract_number")["event_month"].min()
    return (first_event + 1).rename("censored_from_month")


def is_censored_at(
    contracts: pd.Series, months: pd.Series, censored_from: pd.Series
) -> pd.Series:
    """Elementwise: is this (contract, month) at or past the contract's censoring
    boundary? Contracts with no censoring event are never censored (NaT -> False)."""
    boundary = pd.Series(contracts).map(censored_from)
    return boundary.notna() & (pd.Series(months).values >= boundary)


censored_from = censored_from_month(censoring_events)

print(f"{len(censored_from):,} contracts carry a censoring boundary")
print("\nBoundary month (= first event month + 1):")
print(censored_from.value_counts().sort_index().to_string())

# How much of each portfolio month's Stage 3 pool is censored by then. A rising share
# is not a data problem -- it is the share of that month's denominator that Phase C
# must drop rather than score as a clean survivor.
pool_months = stage3_pool.assign(
    portfolio_month=stage3_pool["portfolio_asof"].dt.to_period("M")
)
pool_months["censored"] = is_censored_at(
    pool_months["contract_number"], pool_months["portfolio_month"], censored_from
)
print("\nShare of each portfolio month's Stage 3 pool already censored:")
print(
    pool_months.groupby("portfolio_label")
    .agg(pool=("contract_number", "size"), censored=("censored", "sum"))
    .assign(pct=lambda d: (d["censored"] / d["pool"] * 100).round(2))
    .to_string()
)

### Проверка на восстановления — наблюдением, а не по файлу

В «Приложении №1» **колонки с типом операции нет**. Реальный состав (27.07.2026):

`Контракт · Дни просрочки · КОРЗИНА · Провизии % в LAM · Задолженность без учёта дисконта · Задолженность с учётом дисконта · ОД · Все провизии · Дисконты · Штраф 1860 · Провизии 18770 · Продукт · Тэг`

Это балансовая позиция займа на момент операции — сколько должен, в какой корзине, сколько провизий. Признака «списание vs восстановление» среди них нет, поэтому **по файлу вопрос не решается в принципе**. (Между месяцами состав ещё и плывёт: в 05.2026 «Дисконты» называются «Дисконт / премия» — лишний довод читать только колонку с номером контракта и не привязываться к остальным.)

Зато вопрос решается **наблюдением**. Заём, который действительно ушёл из портфеля, перестаёт появляться в `CL_PORTFOLIO_2`. Вернувшийся — появляется снова. Панель за 12 месяцев у нас есть, так что проверяем прямо: **встречается ли цензурированный контракт в панели на своей границе M+1 или позже?**

Как читать результат:

- **ноль** — все 22 тысячи ушли и не вернулись, цензурирование безопасно, вопрос закрыт;
- **немного (порядка тех самых 64 повторов)** — единичные восстановления; их надо исключить из цензурирования поимённо, а не отменять правило целиком;
- **много** — «Приложение №1» не является чистым списком выходов, и цензурировать по нему нельзя без разделения операций.

Это сильнее любой пометки в файле: мы смотрим, что с займом произошло на самом деле, а не что о нём написали.

In [ ]:
def censoring_reentry_check(
    dpd_panel: pd.DataFrame, censored_from: pd.Series
) -> pd.DataFrame:
    """Censored contracts that still appear in the panel AT OR AFTER their boundary.

    The annexes carry no operation-type column, so "written off" vs "restored" cannot be
    read off the file. It can be observed: a loan that genuinely left stops appearing in
    CL_PORTFOLIO_2. One that reappears did not leave — censoring it from M+1 would
    delete real observation and understate the re-default rate.
    """
    panel = dpd_panel.assign(snap_month=dpd_panel["snap_date"].dt.to_period("M"))
    merged = panel.merge(
        censored_from, left_on="contract_number", right_index=True, how="inner"
    )
    after = merged[merged["snap_month"] >= merged["censored_from_month"]]
    if after.empty:
        return pd.DataFrame(
            columns=["censored_from_month", "months_seen_after", "last_seen", "max_dpd_after"]
        )
    return (
        after.groupby("contract_number")
        .agg(
            censored_from_month=("censored_from_month", "first"),
            months_seen_after=("snap_month", "nunique"),
            last_seen=("snap_month", "max"),
            max_dpd_after=("dpd", "max"),
        )
        .sort_values("months_seen_after", ascending=False)
    )


reentered = censoring_reentry_check(dpd_panel, censored_from)
n_censored = len(censored_from)

print(f"{len(reentered):,} of {n_censored:,} censored contracts "
      f"({len(reentered) / max(n_censored, 1) * 100:.2f}%) still appear in the panel "
      "at or after their censoring boundary")

if reentered.empty:
    print("\n[OK] Nothing came back. The annexes behave as a pure exit list, so treating\n"
          "     every row as a censoring event is safe and the restoration question is\n"
          "     closed empirically -- no operation-type column needed.")
else:
    print("\n[CHECK] These loans did NOT leave the portfolio when the annex says they were\n"
          "        processed. Either they were restored, or the annex lists intentions\n"
          "        rather than completed operations. Censoring them from M+1 would delete\n"
          "        genuine observation, so exclude them by name (see below) instead of\n"
          "        dropping the censoring rule.")
    print(f"\nMonths still observed after the boundary — distribution:")
    print(reentered["months_seen_after"].value_counts().sort_index().to_string())
    print(f"\nTop 15:")
    print(reentered.head(15).to_string())

# Censoring set actually safe to apply: everything that did leave and stayed gone.
censored_from_clean = censored_from.drop(index=reentered.index, errors="ignore")
print(f"\ncensored_from_clean: {len(censored_from_clean):,} contracts "
      f"({n_censored - len(censored_from_clean):,} excluded as re-entering) "
      "-- use THIS in Phase C, not censored_from")

## Next: Phase C (not yet built here)

This notebook covers Phase B of `stage3_safezone_plan.md` — the raw extracts, the 6-month lookback windows, the three-state restructuring classification (`restr_active_pct` / `restr_unknown_pct`), the coverage/usability metric, and the censoring-event load with its coverage audit.

**Run this first and confirm four things before Phase C:**

1. **Row counts** match the SQL-side numbers (the sanity-check cell).
2. **`mean_unknown_pct`** per portfolio month — how much of the restructuring signal is unanswerable. This bounds what the restructured-vs-not split can claim.
3. **`pct_event_dated`** — of the loans that have a restructuring event, how many carry usable grace dates.
4. **`censoring_coverage`** — which panel months are fully covered and which are still `НЕ ПОДТВЕРЖДЕНО`.

**Still open before Phase C can be written** (see the plan's "Locked methodology"):

- **A fixed re-default horizon K.** `@LastAsOf` equals the newest portfolio date, so forward runway ranges from 11 months (08.2025) to zero (07.2026). Scanning "up to the latest available report date" makes the 12 columns non-comparable and pushes the apparent re-default rate down toward the recent end — an elbow picked off that curve is a censoring artifact. Proposal on the table: **K=6**, main matrix restricted to cohorts with full runway (08.2025–01.2026), the rest reported separately and labelled incomplete.
- **Confirm the six write-off-free months** (09, 11·2025 and 01, 02, 03, 07·2026). No rows in the archive, but no confirmation either — so they read `НЕ ПОДТВЕРЖДЕНО` and their re-default rate stays a lower bound. Add them to `WRITEOFF_DID_NOT_OCCUR` once checked and the caveat disappears entirely.
- **Do the Credilogic annexes also contain restorations?** The archive is «списание-**восстановление**» and 64 contracts appear in more than one month. If restorations are mixed in, those loans came back into the portfolio and must not be censored. Only the contract-number column is read today — check with `run_prilozhenie_scan(inspect=True)` whether a type column exists.

**Already settled and encoded here:** censoring starts at **M+1** (`censored_from_month`), the first event wins where a contract repeats, and a month is treated as covered only when it has events **or** a dated confirmation that none occurred — never merely because the file is empty for it. Sales are resolved: they happened in 12.2025 only (БРМ, 27.07.2026), so the other months are covered by that confirmation rather than blind to a missing register.

Phase C (threshold × re-default matrix over n ∈ {0, 3, 7, …, 30}) and Phase D (H1 vs H2) build on top of what's here.

# Phase C — порог безопасной зоны

**ПРОСТЫМ ЯЗЫКОМ:** для каждого порога `n` помечаем займы, у которых DPD не превышал `n` **все 6 месяцев** окна («условно оздоровился»), и смотрим вперёд: вернулся ли он в 3-ю стадию в течение K месяцев. Доля вернувшихся — это и есть re-default rate, по которому выбирается порог.

## Три вещи, которые здесь решаются явно

**1. Горизонт K одинаковый для всех когорт.** Считаем при **K = 3, 6 и 9**. Если рекомендуемый порог при всех трёх примерно один — вывод устойчив, и это сильный аргумент на разговоре с регулятором. Если расходится — лучше узнать это самим.

Цена фиксированного K — часть когорт выпадает: наблюдать K месяцев вперёд можно только там, где эти месяцы есть в данных (панель кончается 07.2026).

| K | Пригодных когорт | Каких |
|---|---|---|
| 3 | 9 | 08.2025 – 04.2026 |
| 6 | 6 | 08.2025 – 01.2026 |
| 9 | 3 | 08.2025 – 10.2025 |

**2. Цензурированные не считаются ни выжившими, ни сорвавшимися.** Если заём продали/списали до конца горизонта и он до этого не сорвался — мы просто не досмотрели, и он **исключается из знаменателя**. Считать его «не сорвался» — значит завысить безопасность порога ровно на тех займах, которые из портфеля выбыли. Используется `censored_from_clean` (вернувшиеся в портфель уже исключены проверкой выше).

**3. Неполное окно наблюдения ≠ чистое окно.** Заём, у которого в 6-месячном окне есть только 2 снимка, не является доказательством шести чистых месяцев. `min_months` по умолчанию требует полного окна; сколько займов отсеяно — печатается, а не замалчивается.

In [ ]:
THRESHOLDS = [0, 3, 5, 7, 10, 15, 20, 25, 30]
HORIZONS = [3, 6, 9]
STAGE3_DPD_TRIGGER = 91          # DPD at which a loan re-qualifies for Stage 3
PANEL_LAST_MONTH = dpd_panel["snap_date"].max().to_period("M")

def _norm_category(s: pd.Series) -> pd.Series:
    """'3' whether the CSV round-trip handed category back as int, float or text.

    category is numeric-as-text in CL_PORTFOLIO_2, and a single missing value anywhere
    in the column is enough for pandas to read the whole thing as float64. After that
    astype("string") yields "3.0", every comparison to "3" is False, and the failure is
    silent: `exits` matches every row, `returned` matches none, and the category-based
    cross-check reports 0.0% as though that were a measurement rather than an empty set.
    """
    return s.astype("string").str.strip().str.replace(r"\.0+$", "", regex=True)


_panel = dpd_panel.assign(
    snap_month=dpd_panel["snap_date"].dt.to_period("M"),
    category_str=_norm_category(dpd_panel["category"]),
)

_cat_seen = set(_panel["category_str"].dropna().unique())
if "3" not in _cat_seen:
    raise AssertionError(
        "В нормализованном category нет значения '3' — сравнение по стадии не сработает "
        f"ни на одной строке. Найдено: {sorted(_cat_seen)[:10]}. Кросс-проверка вернула "
        "бы 0.0% по всем когортам, и это выглядело бы как вывод, а не как поломка."
    )

# Last month each contract appears in the panel at all. A contract whose last month is
# before the panel's own end has LEFT — repaid and closed, sold, written off, or moved
# somewhere this extract does not see. censoring_events.csv only knows about the
# Credilogic write-off annexes, so every other kind of departure is invisible to it.
last_seen_month = _panel.groupby("contract_number")["snap_month"].max()
vanished_from = (last_seen_month[last_seen_month < PANEL_LAST_MONTH] + 1).rename(
    "vanished_from"
)


def _earliest(a: pd.Series, b: pd.Series) -> pd.Series:
    """Element-wise earlier of two month Series, either of which may be NaT."""
    out = a.copy()
    take_b = b.notna() & (a.isna() | (b < a))
    out[take_b] = b[take_b]
    return out


def _event_months(events: pd.Series, index) -> pd.PeriodIndex:
    """Align an event-month Series onto `index`, missing entries as NaT.

    reindex rather than Index.map: map() raises on an EMPTY Period mapper (pandas tries
    to coerce it to float64), and a cohort where the event never fires is an ordinary
    case, not an error — it must not take the whole run down."""
    if len(events) == 0:
        return pd.PeriodIndex([pd.NaT] * len(index), freq="M")
    return pd.PeriodIndex(events.reindex(index), freq="M")


def build_cohort(asof: pd.Timestamp, label: str) -> pd.DataFrame:
    """Per contract in this month's Stage 3 pool: the window's worst DPD, how many
    months of it were observed, and WHEN it re-defaulted under each of two definitions.

    Two, because the obvious one is degenerate here. The plan's original wording —
    "the first later month category returns to '3'" — assumes the loan left Stage 3
    first. These loans did not: they sit in Stage 3 at `asof` precisely because the
    current cure rule has not released them, and category stays '3' at asof+1. Read
    literally, every flagged loan would score as a re-default at M+1.

      redefault_dpd  (primary) -- first later month with DPD >= STAGE3_DPD_TRIGGER.
          This is the counterfactual actually being asked: *if we had cured this loan,
          would it have deteriorated back to Stage 3 severity?* It does not depend on
          the bank's current cure mechanics at all, which is what makes it usable as
          evidence for changing them.
      redefault_category (cross-check) -- the loan must first LEAVE Stage 3
          (category != '3'), then return. Measures loans genuinely cured under today's
          rule that came back. Real, but on a smaller and self-selected population —
          it can only see loans the current rule already released.
    """
    pool = stage3_pool.loc[stage3_pool["portfolio_label"] == label, "contract_number"].unique()
    window = build_lookback_dpd(dpd_panel, asof)
    window = window[window["contract_number"].isin(pool)]

    prof = window.groupby("contract_number")["dpd"].agg(
        max_dpd="max", months_observed="count"
    )
    asof_month = asof.to_period("M")
    after = _panel[_panel["snap_month"] > asof_month]

    by_dpd = (
        after[after["dpd"] >= STAGE3_DPD_TRIGGER]
        .groupby("contract_number")["snap_month"].min()
    )
    exits = (
        after[after["category_str"] != "3"]
        .groupby("contract_number")["snap_month"].min().rename("exit_month")
    )
    returned = after[after["category_str"] == "3"].merge(
        exits, left_on="contract_number", right_index=True
    ).reset_index(drop=True)   # merge puts contract_number on the index too -> ambiguous
    by_category = (
        returned[returned["snap_month"] > returned["exit_month"]]
        .groupby("contract_number")["snap_month"].min()
    )

    prof["redefault_dpd"] = _event_months(by_dpd, prof.index)
    prof["redefault_category"] = _event_months(by_category, prof.index)
    cens = pd.Series(_event_months(censored_from_clean, prof.index), index=prof.index)
    van = pd.Series(_event_months(vanished_from, prof.index), index=prof.index)
    prof["censored_from"] = cens
    # The pessimistic boundary: treat ANY departure from the panel as an exit we did
    # not get to observe. Not a correction of the line above — the opposite bound. A
    # loan that repaid in full and closed also disappears, and censoring that one
    # deletes a genuine survivor. The two columns bracket the answer; neither is it.
    prof["censored_from_silent"] = _earliest(cens, van)
    restr = restr_state_pct_by_portfolio[label]
    prof["restr_active_pct"] = restr["restr_active_pct"].reindex(prof.index)
    prof["restr_unknown_pct"] = restr["restr_unknown_pct"].reindex(prof.index)
    return prof


def redefault_stats(
    prof: pd.DataFrame,
    asof_month,
    n: int,
    K: int,
    event_col: str = "redefault_dpd",
    min_months: int = LOOKBACK_MONTHS,
    censor_col: str = "censored_from",
) -> dict:
    """Outcome for threshold n over horizon K on one cohort.

    Three outcomes, and the third is the one that is easy to get wrong:
      redefaulted  -- the event happened inside the horizon (numerator + denominator)
      survived     -- observable for the full K months and no event (denominator)
      censored_out -- sold/written off before the horizon ended, no event first.
                      NEITHER. We did not watch long enough to know. Scoring these as
                      survivors is exactly how a threshold is made to look safe using
                      the loans that left the portfolio.
    """
    flagged = prof[(prof["max_dpd"] <= n) & (prof["months_observed"] >= min_months)]
    horizon_end = asof_month + K

    cens_last = flagged[censor_col] - 1             # last month still observable
    observed_until = cens_last.where(
        cens_last.notna() & (cens_last < horizon_end), horizon_end
    )

    event = flagged[event_col]
    redefaulted = event.notna() & (event <= observed_until)
    completed = observed_until >= horizon_end
    in_denominator = redefaulted | completed
    denom = int(in_denominator.sum())

    return {
        "flagged": int(len(flagged)),
        "redefaulted": int(redefaulted.sum()),
        "survived": int((completed & ~redefaulted).sum()),
        "censored_out": int((~in_denominator).sum()),
        "denominator": denom,
        # NaN, never 0.0 -- "no loans qualified" is not "nobody re-defaulted".
        "rate": (int(redefaulted.sum()) / denom) if denom else np.nan,
    }


cohort_months = {
    row["portfolio_label"]: row["asof_date"].to_period("M")
    for _, row in report_dates.iterrows()
}
cohorts = {
    row["portfolio_label"]: build_cohort(row["asof_date"], row["portfolio_label"])
    for _, row in report_dates.iterrows()
}


def eligible_cohorts(K: int) -> list:
    """Cohorts with a full K months of forward data. Anything shorter would be measured
    on a different runway and is not comparable — the entire point of fixing K."""
    return [lbl for lbl, m in cohort_months.items() if m + K <= PANEL_LAST_MONTH]


print(f"Panel ends {PANEL_LAST_MONTH}; {len(cohorts)} cohorts built\n")
for K in HORIZONS:
    elig = sorted(eligible_cohorts(K), key=lambda l: cohort_months[l])
    print(f"K={K}: {len(elig)} eligible cohort(s) — {', '.join(elig) or 'NONE'}")

diag = pd.DataFrame({
    "pool": {l: len(p) for l, p in cohorts.items()},
    "incomplete_window": {
        l: int((p["months_observed"] < LOOKBACK_MONTHS).sum()) for l, p in cohorts.items()
    },
    "ever_redefault_dpd": {
        l: int(p["redefault_dpd"].notna().sum()) for l, p in cohorts.items()
    },
    "ever_redefault_category": {
        l: int(p["redefault_category"].notna().sum()) for l, p in cohorts.items()
    },
}).sort_index()
print("\nPer cohort — pool size, loans dropped for an incomplete window, and how many "
      "ever re-default\nunder each definition (before any threshold or horizon is applied):")
print(diag.to_string())
print(
    "\nIf ever_redefault_category is far smaller than ever_redefault_dpd, that is not a\n"
    "bug: it is the current cure rule releasing very few loans, which is the reason this\n"
    "study exists. The DPD definition is the primary one; category is the cross-check."
)

In [ ]:
def redefault_matrix(
    K: int, event_col: str = "redefault_dpd", min_months: int = LOOKBACK_MONTHS,
    censor_col: str = "censored_from",
) -> dict:
    """threshold × cohort matrices for one horizon: rate, denominator, censored-out.

    All three are returned together on purpose. A rate without its denominator invites
    reading 100% off two loans, and the censored count shows how much of the cohort the
    answer had to be built without.
    """
    labels = sorted(eligible_cohorts(K), key=lambda l: cohort_months[l])
    rate = pd.DataFrame(index=THRESHOLDS, columns=labels, dtype=float)
    denom = pd.DataFrame(index=THRESHOLDS, columns=labels, dtype=float)
    cens = pd.DataFrame(index=THRESHOLDS, columns=labels, dtype=float)
    pooled = []

    for n in THRESHOLDS:
        red_sum = den_sum = 0
        for lbl in labels:
            s = redefault_stats(
                cohorts[lbl], cohort_months[lbl], n, K, event_col, min_months, censor_col
            )
            rate.loc[n, lbl] = s["rate"]
            denom.loc[n, lbl] = s["denominator"]
            cens.loc[n, lbl] = s["censored_out"]
            red_sum += s["redefaulted"]
            den_sum += s["denominator"]
        pooled.append({"threshold": n, "redefaulted": red_sum, "denominator": den_sum,
                       "rate": (red_sum / den_sum) if den_sum else np.nan})

    rate.index.name = denom.index.name = cens.index.name = "threshold_n"
    return {"K": K, "event_col": event_col, "cohorts": labels, "rate": rate,
            "denominator": denom, "censored_out": cens,
            "pooled": pd.DataFrame(pooled).set_index("threshold")}


matrices = {K: redefault_matrix(K) for K in HORIZONS}
matrices_cat = {K: redefault_matrix(K, event_col="redefault_category") for K in HORIZONS}

# Table view — also the accessibility relief for the charts below, so these are printed,
# not merely computed.
for K in HORIZONS:
    m = matrices[K]
    print(f"\n{'=' * 78}\nK = {K} мес.   ({len(m['cohorts'])} сопоставимых когорт)   "
          f"событие: DPD >= {STAGE3_DPD_TRIGGER}\n{'=' * 78}")
    if not m["cohorts"]:
        print("  НЕТ пригодных когорт — панель короче горизонта.")
        continue
    print("\nre-default rate, % (NaN = ни один заём не прошёл порог):")
    print((m["rate"] * 100).round(1).to_string())
    print("\nзнаменатель (досмотрены весь горизонт, либо сорвались до выбытия):")
    print(m["denominator"].astype("Int64").to_string())
    print("\nисключены как цензурированные (выбыли до конца горизонта):")
    print(m["censored_out"].astype("Int64").to_string())

print(f"\n\n{'=' * 78}\nPOOLED — чувствительность к горизонту K\n{'=' * 78}")
pooled_rates = pd.DataFrame(
    {f"K={K}": matrices[K]["pooled"]["rate"] * 100 for K in HORIZONS}).round(1)
pooled_denoms = pd.DataFrame(
    {f"K={K}": matrices[K]["pooled"]["denominator"] for K in HORIZONS}).astype("Int64")
print("\nre-default rate (DPD-определение), %:")
print(pooled_rates.to_string())
print("\nзнаменатель:")
print(pooled_denoms.to_string())

print(f"\n\n{'=' * 78}\nПЕРЕКРЁСТНАЯ ПРОВЕРКА — определение по возврату category в '3'"
      f"\n{'=' * 78}")
pooled_cat = pd.DataFrame(
    {f"K={K}": matrices_cat[K]["pooled"]["rate"] * 100 for K in HORIZONS}).round(1)
print("\nre-default rate (category-определение), %:")
print(pooled_cat.to_string())
print(
    "\nЭто заведомо меньшая и самоотобранная популяция: category-определение видит\n"
    "только те займы, которые ДЕЙСТВУЮЩЕЕ правило уже выпустило из 3-й стадии. Если\n"
    "цифры сильно ниже — это не ошибка, а ровно та проблема, ради которой затеян\n"
    "пересмотр правила. Решение принимается по DPD-определению; category — контроль\n"
    "направления, а не источник порога."
)
print(
    "\nЧитать колонки в сравнении, а не только вниз. Если излом приходится на один и тот\n"
    "же n при K=3, 6 и 9 — порог является свойством портфеля. Если он ездит вместе с K —\n"
    "это свойство длины наблюдения, и n* обязан указываться вместе с горизонтом."
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
K_COLOR = {3: "#2a78d6", 6: "#eb6834", 9: "#1baf7a"}   # validated all-pairs, light mode
MIN_BASE = 30      # below this many loans a rate is noise, and is drawn hollow

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",  # solid hairline
    "font.size": 9, "axes.titlesize": 10.5, "legend.frameon": False,
})


def _style(ax):
    ax.grid(True, axis="y", zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_linewidth(0.8)


def _plot_series(ax, x, y, base, color, label=None, lw=2.0, z=3):
    """Line + markers, with points backed by fewer than MIN_BASE loans drawn hollow —
    a rate off 4 loans must not look like the same evidence as a rate off 4000."""
    ax.plot(x, y, color=color, linewidth=lw, zorder=z, label=label,
            solid_capstyle="round")
    solid = [b >= MIN_BASE for b in base]
    for xi, yi, ok in zip(x, y, solid):
        if pd.isna(yi):
            continue
        ax.plot(xi, yi, marker="o", markersize=5, zorder=z + 1,
                color=color if ok else SURFACE,
                markeredgecolor=color, markeredgewidth=1.5)


# ── Figure 1 — one panel per horizon: every cohort, plus the pooled line ────────────
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(13, 4.2), sharey=True)
axes = np.atleast_1d(axes)

for ax, K in zip(axes, HORIZONS):
    m = matrices[K]
    _style(ax)
    if not m["cohorts"]:
        ax.text(0.5, 0.5, f"K={K}\nнет пригодных когорт\n(панель короче горизонта)",
                ha="center", va="center", color=MUTED, transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        continue

    for lbl in m["cohorts"]:
        # Cohorts stay recessive — they are spread, not identity, so no 9 hues. Points
        # below MIN_BASE are masked out entirely: a cohort with two qualifying loans
        # prints 100% or 0% and would otherwise draw a vertical streak across the panel
        # that carries no information. The tables above still show every cell.
        series = m["rate"][lbl] * 100
        ax.plot(THRESHOLDS, series.where(m["denominator"][lbl] >= MIN_BASE),
                color=GRID, linewidth=1.0, zorder=1)
    _plot_series(ax, THRESHOLDS, m["pooled"]["rate"] * 100,
                 m["pooled"]["denominator"], K_COLOR[K], lw=2.0)

    last = m["pooled"]["rate"].dropna()
    if not last.empty:                            # selective direct label, not every point
        ax.annotate(f"K={K}\n{last.iloc[-1] * 100:.1f}%",
                    xy=(last.index[-1], last.iloc[-1] * 100),
                    xytext=(-4, 10), textcoords="offset points",
                    color=K_COLOR[K], fontweight="bold", ha="right")
    ax.set_title(f"K = {K} мес.  ·  {len(m['cohorts'])} когорт", loc="left")
    ax.set_xlabel("порог n (DPD ≤ n все 6 мес.)")

axes[0].set_ylabel("re-default rate, %")
fig.legend(handles=[
    Line2D([], [], color=GRID, lw=1.0, label="отдельная когорта"),
    Line2D([], [], color=INK_2, lw=2.0, label="pooled (все когорты)"),
    Line2D([], [], marker="o", color=SURFACE, markeredgecolor=INK_2, markeredgewidth=1.5,
           lw=0, label=f"база < {MIN_BASE} займов"),
], loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("Re-default rate по порогу n — по каждому горизонту наблюдения",
             x=0.008, ha="left", fontsize=12, color=INK, fontweight="bold")
fig.tight_layout(rect=[0, 0.04, 1, 0.95])
plt.show()


# ── Figure 2 — the decision view: does the elbow move with K? ───────────────────────
fig, (ax_rate, ax_base) = plt.subplots(
    2, 1, figsize=(8.5, 6.4), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)
for ax in (ax_rate, ax_base):
    _style(ax)

for K in HORIZONS:
    p = matrices[K]["pooled"]
    if p["denominator"].sum() == 0:
        continue
    _plot_series(ax_rate, THRESHOLDS, p["rate"] * 100, p["denominator"],
                 K_COLOR[K], label=f"K = {K} мес.")
    valid = p["rate"].dropna()
    if not valid.empty:
        ax_rate.annotate(f"K={K}", xy=(valid.index[-1], valid.iloc[-1] * 100),
                         xytext=(6, 0), textcoords="offset points",
                         color=K_COLOR[K], fontweight="bold", va="center")
    ax_base.plot(THRESHOLDS, p["denominator"], color=K_COLOR[K], linewidth=1.6)

ax_rate.set_ylabel("re-default rate, %")
ax_rate.set_title("Порог n против доли сорвавшихся — три горизонта на одной шкале",
                  loc="left", fontweight="bold")
ax_rate.legend(loc="upper left")
# Base counts get their own panel rather than a second y-axis: two scales on one plot
# is the single most misread chart there is.
ax_base.set_ylabel("знаменатель,\nзаймов")
ax_base.set_xlabel("порог n (DPD ≤ n все 6 мес.)")
# Log scale: counts run from single digits to five, and on a linear axis the trust
# threshold would sit indistinguishably on the baseline — which is the one thing this
# panel exists to make visible.
ax_base.set_yscale("log")
ax_base.axhline(MIN_BASE, color=MUTED, linewidth=0.8, zorder=1)
ax_base.annotate(f"порог доверия — {MIN_BASE} займов", xy=(THRESHOLDS[-1], MIN_BASE),
                 xytext=(-2, 5), textcoords="offset points", color=MUTED, fontsize=8,
                 ha="right")
fig.tight_layout()
plt.show()

print("Полые точки = база меньше "
      f"{MIN_BASE} займов: доля посчитана, но опираться на неё нельзя.\n"
      "Числа — в таблицах выше; график показывает форму, решение принимается по обоим.")

## Phase C — разрез по реструктуризации (и почему он может не состояться)

Последний пункт Phase C: развернуть матрицу порогов по состоянию реструктуризации
и показать долю `unknown` **в каждой ячейке**.

**Сегмент займа за окно — по тому же трёхзначному правилу, что и месяц.** Приоритет
сверху вниз, первое сработавшее выигрывает:

| Сегмент | Правило | Почему так |
|---|---|---|
| `restructured` | хотя бы один месяц окна = `active` | подтверждённое покрытие — это установленный факт, и его не отменяет то, что другие месяцы нечитаемы |
| `unknown` | хотя бы один месяц = `unknown` (или состояние вообще не посчитано) | достаточно одного неотвечаемого месяца, чтобы «не был под реструктуризацией» стало недоказуемым |
| `not_restructured` | все наблюдённые месяцы = `not_active` | единственный случай, когда отрицание опирается на данные, а не на их отсутствие |

**Разрез не всегда даёт право на вывод, и это здесь считается, а не подразумевается.**
Зафиксированная методология требует показывать долю `unknown` рядом со сравнением:
сегмент с существенной долей нечитаемых окон не подтверждает вывод о реструктуризации
**ни в одну, ни в другую сторону**. «Существенная» переведена в число —
`UNKNOWN_MATERIALITY`, — чтобы порог был выбран до того, как станут видны цифры, а не
подогнан под них. Ячейки, не прошедшие проверку, печатаются с явным вердиктом, а не
исчезают: «сравнение не поддержано данными» — это тоже результат, и на разговоре с
регулятором он честнее умолчания.

Оба барьера независимы: `unknown` может быть мал, но обе базы малы — тогда вердикт
тоже отрицательный, просто по другой причине.

In [ ]:
# Доля unknown, выше которой разрез перестаёт быть свидетельством. Выбрано ДО
# просмотра результата: 10% нечитаемых окон уже достаточно, чтобы перевернуть
# сравнение двух ставок, отличающихся на несколько п.п.
UNKNOWN_MATERIALITY = 0.10
SEGMENTS = ["restructured", "not_restructured", "unknown"]


def restr_segment(prof: pd.DataFrame) -> pd.Series:
    """Сегмент займа за окно: 'restructured' / 'not_restructured' / 'unknown'.

    NaN (состояние не посчитано — займа нет в окне) идёт в 'unknown', а не в
    'not_restructured': это ровно то схлопывание, которое методология запрещает.
    Отсутствие ответа и отрицательный ответ — разные вещи, и здесь они не
    смешиваются даже ценой более крупного бесполезного сегмента.
    """
    active = prof["restr_active_pct"].fillna(0) > 0
    unknown = prof["restr_unknown_pct"].fillna(0) > 0
    not_computed = prof["restr_active_pct"].isna() & prof["restr_unknown_pct"].isna()
    return pd.Series(
        np.where(active, "restructured",
                 np.where(unknown | not_computed, "unknown", "not_restructured")),
        index=prof.index, name="restr_segment",
    )


segments_by_cohort = {lbl: restr_segment(prof) for lbl, prof in cohorts.items()}

mix = pd.DataFrame(
    {lbl: seg.value_counts(normalize=True).reindex(SEGMENTS).fillna(0) * 100
     for lbl, seg in segments_by_cohort.items()}
).T.round(1)
mix["loans"] = {lbl: len(seg) for lbl, seg in segments_by_cohort.items()}
mix = mix.sort_index()

print("Состав пула по сегментам реструктуризации, % займов (до порога и горизонта):")
print(mix.to_string())
print(
    f"\nunknown в среднем по месяцам: {mix['unknown'].mean():.1f}%"
    f"  (порог существенности — {UNKNOWN_MATERIALITY * 100:.0f}%)"
)
print(
    "Читать до таблиц ниже: если unknown здесь уже велик, разрез не спасёт никакой\n"
    "порог n — сравнивать будет попросту нечего, и это вывод про заполненность\n"
    "grace-дат в restructuring_v2, а не про реструктуризацию как таковую."
)

In [ ]:
def redefault_split_matrix(
    K: int, event_col: str = "redefault_dpd", min_months: int = LOOKBACK_MONTHS
) -> dict:
    """Матрица порог × когорта, развёрнутая по сегментам реструктуризации.

    Возвращает per-cell rate/denominator по каждому сегменту И долю unknown в той же
    ячейке — иначе разрез читается как свидетельство там, где он держится на
    нечитаемых окнах.
    """
    labels = sorted(eligible_cohorts(K), key=lambda l: cohort_months[l])
    frame = lambda: pd.DataFrame(index=THRESHOLDS, columns=labels, dtype=float)
    rate = {s: frame() for s in SEGMENTS}
    denom = {s: frame() for s in SEGMENTS}
    unknown_share = frame()
    pooled = []

    for n in THRESHOLDS:
        tot = {s: {"redefaulted": 0, "denominator": 0, "flagged": 0} for s in SEGMENTS}
        for lbl in labels:
            prof, seg = cohorts[lbl], segments_by_cohort[lbl]
            flagged_here = {}
            for s in SEGMENTS:
                sub = prof[seg == s]
                if sub.empty:
                    flagged_here[s] = 0
                    continue
                st = redefault_stats(
                    sub, cohort_months[lbl], n, K, event_col, min_months
                )
                rate[s].loc[n, lbl] = st["rate"]
                denom[s].loc[n, lbl] = st["denominator"]
                flagged_here[s] = st["flagged"]
                tot[s]["redefaulted"] += st["redefaulted"]
                tot[s]["denominator"] += st["denominator"]
                tot[s]["flagged"] += st["flagged"]
            total_flagged = sum(flagged_here.values())
            # Доля unknown считается от ПОМЕЧЕННЫХ порогом займов, а не от всего пула:
            # вопрос не «сколько unknown в портфеле», а «на чём стоит именно эта ячейка».
            unknown_share.loc[n, lbl] = (
                flagged_here["unknown"] / total_flagged if total_flagged else np.nan
            )

        row = {"threshold": n}
        for s in SEGMENTS:
            d = tot[s]["denominator"]
            row[f"rate_{s}"] = (tot[s]["redefaulted"] / d) if d else np.nan
            row[f"den_{s}"] = d
            row[f"flagged_{s}"] = tot[s]["flagged"]
        flagged_all = sum(tot[s]["flagged"] for s in SEGMENTS)
        row["unknown_share"] = (
            tot["unknown"]["flagged"] / flagged_all if flagged_all else np.nan
        )
        pooled.append(row)

    for df in (*rate.values(), *denom.values(), unknown_share):
        df.index.name = "threshold_n"
    return {"K": K, "event_col": event_col, "cohorts": labels, "rate": rate,
            "denominator": denom, "unknown_share": unknown_share,
            "pooled": pd.DataFrame(pooled).set_index("threshold")}


def split_verdict(row) -> str:
    """Можно ли вообще что-то сказать про эту строку — и если нет, то почему именно.

    Две причины отказа, и их не следует смешивать: 'нет базы' чинится накоплением
    когорт, 'unknown' — только заполнением grace-дат в источнике.
    """
    if row["unknown_share"] >= UNKNOWN_MATERIALITY:
        return f"НЕ ПОДДЕРЖАНО (unknown {row['unknown_share'] * 100:.0f}%)"
    thin = [s for s in ("restructured", "not_restructured") if row[f"den_{s}"] < MIN_BASE]
    if thin:
        return "НЕТ БАЗЫ (" + ", ".join(f"{s}: {int(row[f'den_{s}'])}" for s in thin) + ")"
    delta = (row["rate_restructured"] - row["rate_not_restructured"]) * 100
    if pd.isna(delta):
        return "НЕТ ДАННЫХ"
    return f"сравнимо: {delta:+.1f} п.п. у реструктурированных"


splits = {K: redefault_split_matrix(K) for K in HORIZONS}

for K in HORIZONS:
    sp = splits[K]
    print(f"\n{'=' * 84}\nK = {K} мес.  —  разрез по реструктуризации "
          f"({len(sp['cohorts'])} когорт)\n{'=' * 84}")
    if not sp["cohorts"]:
        print("  НЕТ пригодных когорт — панель короче горизонта.")
        continue

    p = sp["pooled"]
    view = pd.DataFrame({
        "restr_%": (p["rate_restructured"] * 100).round(1),
        "restr_n": p["den_restructured"].astype("Int64"),
        "not_restr_%": (p["rate_not_restructured"] * 100).round(1),
        "not_restr_n": p["den_not_restructured"].astype("Int64"),
        "unknown_%_от_помеченных": (p["unknown_share"] * 100).round(1),
        "вердикт": p.apply(split_verdict, axis=1),
    })
    print("\nPOOLED по всем сопоставимым когортам:")
    print(view.to_string())

    print("\nДоля unknown ПО ЯЧЕЙКАМ (порог × когорта), %:")
    print((sp["unknown_share"] * 100).round(1).to_string())

supported = {
    K: [n for n, r in splits[K]["pooled"].iterrows()
        if not split_verdict(r).startswith(("НЕ ПОДДЕРЖАНО", "НЕТ"))]
    for K in HORIZONS
}
print(f"\n\n{'=' * 84}\nИТОГ РАЗРЕЗА\n{'=' * 84}")
for K in HORIZONS:
    ok = supported[K]
    print(f"K={K}: порогов с допустимым сравнением — {len(ok)}"
          + (f" ({', '.join(map(str, ok))})" if ok else " — ни одного"))
print(
    "\nЕсли допустимых порогов нет ни при одном K, вывод формулируется так: данные не\n"
    "позволяют сказать, что реструктурированные займы срываются чаще или реже. Это НЕ\n"
    "равно «разницы нет» — разница может быть любой, мы её не измерили. Подпирается\n"
    "заполнением grace-дат в restructuring_v2, а не выбором другого порога n."
)

## Сколько стоит невидимое выбытие — вилка, а не поправка

`censoring_events.csv` знает только про списания из «Приложения №1». Всё остальное, чем заём может уйти из портфеля, для него не существует — в том числе **декабрьские продажи** (`Prodaja&Proschenie_12_2025`, 21 341 контракт), которых в файле нет. Таблица покрытия при этом пишет «полное», и она не врёт: она проверяет наличие источника, а не то, что его строки доехали до множества цензурирования.

Заём без границы и без события уходит в знаменатель **выжившим**. Отсюда вопрос: насколько это смещает ставку?

**Ответ — вилка из двух матриц, а не одно исправленное число.** Считать всякое исчезновение выбытием нельзя: заём, полностью погашенный и закрытый, исчезает точно так же, и цензурировать его — значит вычеркнуть настоящего выжившего. Поэтому:

| Граница | Что предполагает | Куда смещает |
|---|---|---|
| `censored_from` (как считали) | исчез без записи → **выжил** | ставка занижена — **нижняя оценка** |
| `censored_from_silent` | исчез без записи → **не досмотрели** | ставка завышена — **верхняя оценка** |

Истина между ними, и ближе к нижней в той мере, в какой исчезновения — это погашения, а не продажи. Если вилка узкая, вопрос закрыт и продажи можно не грузить вовсе. Если широкая — грузить обязательно, и тогда `Prodaja&Proschenie_12_2025` идёт в `censoring_events` отдельным источником с датой и атрибуцией, как продажи и списания.

In [ ]:
MATRIX_BOUNDS = {"нижняя (как есть)": "censored_from",
                 "верхняя (всякий уход цензурирован)": "censored_from_silent"}

silent_only = sorted(set(vanished_from.index) - set(censored_from_clean.index))
print(f"Контрактов, исчезнувших из панели: {len(vanished_from):,}")
print(f"  из них с записью о цензурировании: "
      f"{len(vanished_from) - len(silent_only):,}")
print(f"  БЕЗ записи — сейчас засчитываются выжившими: {len(silent_only):,}")

bounds = {}
for K in HORIZONS:
    lo = matrices[K]["pooled"]
    hi = redefault_matrix(K, censor_col="censored_from_silent")["pooled"]
    bounds[K] = pd.DataFrame({
        "нижняя_%": (lo["rate"] * 100).round(2),
        "верхняя_%": (hi["rate"] * 100).round(2),
        "ширина_пп": ((hi["rate"] - lo["rate"]) * 100).round(2),
        "знам_нижн": lo["denominator"].astype("Int64"),
        "знам_верх": hi["denominator"].astype("Int64"),
    })

for K in HORIZONS:
    print(f"\n{'=' * 66}\nK = {K} мес.\n{'=' * 66}")
    print(bounds[K].to_string())

worst = max(bounds[K]["ширина_пп"].max() for K in HORIZONS)
print(f"\n\nСамая широкая вилка по всем порогам и горизонтам: {worst:.2f} пп")
print(
    "Как это читать: вилка — это НЕ погрешность и не доверительный интервал. Это два\n"
    "разных допущения о том, что случилось с исчезнувшими займами, и правда лежит\n"
    "между ними ровно в той пропорции, в какой исчезновения были погашениями, а не\n"
    "продажами. Узкая вилка означает, что вопрос не влияет на выбор порога; широкая —\n"
    "что до выбора порога надо загрузить реестр продаж."
)

# Phase D — сколько это в тенге

Phase C отвечала «безопасен ли порог». Здесь считается то, что просил Дамир 17.07: **сколько займов и на какую сумму** освобождает смягчение, против оценки РБ в **12 млрд ₸**.

**Целевое множество — не то же самое, что «помеченные» в Phase C.** Займы с `max_dpd = 0` за окно оздоровятся и по действующему строгому правилу; смягчение их не освобождает, и включать их в сумму против 12 млрд нельзя. Поэтому кандидат — это `max_dpd ∈ [1, n]`: срыв был, но мелкий. Плюс просрочка должна быть погашена **сейчас** (`dpd_asof ≤ CURE_ENTRY_DPD`), иначе речь не об оздоровлении.

**Две гипотезы правила.**

- **H1 (прямая линия)** — `DPD ≤ n` все месяцы окна. Простое, объяснимое, проверяемое.
- **H2 (нисходящий тренд)** — DPD не растёт от месяца к месяцу, безотносительно уровня. По плану — «regardless of whether it ever hit zero», и ровно так и реализовано. Но в этом виде правило пропускает заём 500 → 400 → 300: тренд идеальный, оздоровления нет. Поэтому рядом считается **H2+** — тот же тренд плюс то же требование погашенной просрочки, что и в H1. Обе цифры на столе; выбор между ними — решение, а не деталь реализации.

In [ ]:
RB_ESTIMATE_KZT = 12e9
CURE_ENTRY_DPD = 5              # «просрочка погашена сейчас» — параметр Дамира
CANDIDATE_THRESHOLDS = [1, 3, 5, 7, 10, 30]
CANDIDATE_WINDOWS = [3, 6]      # чувствительность по окну, как просили

_dup_pool = stage3_pool.duplicated(subset=["portfolio_label", "contract_number"]).sum()
if _dup_pool:
    raise AssertionError(
        f"stage3_pool: {_dup_pool:,} дублей по (portfolio_label, contract_number). "
        "Остаток в ₸ ниже посчитался бы с двойным учётом, а reindex упал бы с "
        "сообщением про дубли в индексе, из которого причина не читается. "
        "Разберитесь в §1 stage3_safezone_rolling_extract.sql до того, как "
        "называть цифру."
    )

_pool_money = (
    stage3_pool.assign(
        balance=lambda d: pd.to_numeric(d["balance"], errors="coerce"),
        provisions=lambda d: pd.to_numeric(d["provisions_total"], errors="coerce"),
    )
    .set_index(["portfolio_label", "contract_number"])[["balance", "provisions"]]
)
_bad_money = _pool_money["balance"].isna().sum()
if _bad_money:
    print(f"[ВНИМАНИЕ] {_bad_money:,} строк пула без числового balance — они дадут 0 ₸ "
          f"в суммах ниже, а не молча выпадут из счётчиков.")


def build_candidates(asof: pd.Timestamp, label: str, window_months: int) -> pd.DataFrame:
    """Профиль окна на один портфельный месяц + деньги на дату среза."""
    pool = stage3_pool.loc[stage3_pool["portfolio_label"] == label, "contract_number"]
    w = build_lookback_dpd(dpd_panel, asof, window_months)
    w = w[w["contract_number"].isin(set(pool))].sort_values(["contract_number", "snap_date"])

    g = w.groupby("contract_number")["dpd"]
    prof = pd.DataFrame({
        "max_dpd": g.max(),
        "months_observed": g.count(),
        # is_monotonic_decreasing = НЕвозрастающая (равенства допускаются) — ровно то
        # определение тренда, что зафиксировано в плане, несмотря на имя метода.
        "non_increasing": g.apply(lambda s: s.is_monotonic_decreasing),
    })
    prof["delinquent_months"] = (
        w[w["dpd"] > 0].groupby("contract_number")["dpd"].count().reindex(prof.index).fillna(0).astype(int)
    )
    prof["dpd_asof"] = (
        w[w["snap_date"] == asof].set_index("contract_number")["dpd"].reindex(prof.index)
    )
    money = _pool_money.loc[label].reindex(prof.index)
    prof[["balance", "provisions"]] = money[["balance", "provisions"]].fillna(0.0)
    return prof[prof["months_observed"] >= window_months]


def rule_sets(prof: pd.DataFrame, n: int) -> dict:
    """Три популяции: H1, H2 как записано, H2+ с тем же условием погашенной просрочки."""
    cleared = prof["dpd_asof"] <= CURE_ENTRY_DPD
    slipped = prof["max_dpd"].between(1, n)          # =0 оздоровится и по строгому правилу
    return {
        "H1": prof[slipped & cleared],
        "H2": prof[prof["non_increasing"] & (prof["max_dpd"] >= 1)],
        "H2+": prof[prof["non_increasing"] & (prof["max_dpd"] >= 1) & cleared],
    }


def money_row(sub: pd.DataFrame) -> dict:
    return {"займов": len(sub),
            "остаток_млрд": sub["balance"].sum() / 1e9,
            "провизии_млрд": sub["provisions"].sum() / 1e9}


CANDIDATE_LABEL = report_dates.loc[report_dates["asof_date"].idxmax(), "portfolio_label"]
CANDIDATE_ASOF = report_dates["asof_date"].max()
print(f"Список кандидатов строится на {CANDIDATE_ASOF:%d.%m.%Y} ({CANDIDATE_LABEL})\n")

for wm in CANDIDATE_WINDOWS:
    prof = build_candidates(CANDIDATE_ASOF, CANDIDATE_LABEL, wm)
    rows = []
    for n in CANDIDATE_THRESHOLDS:
        sets_ = rule_sets(prof, n)
        row = {"n": n}
        for name, sub in sets_.items():
            m = money_row(sub)
            row[f"{name}_займов"] = m["займов"]
            row[f"{name}_млрд"] = round(m["остаток_млрд"], 2)
        both = sets_["H1"].index.intersection(sets_["H2+"].index)
        row["пересечение"] = len(both)
        rows.append(row)
    table = pd.DataFrame(rows).set_index("n")
    print(f"{'=' * 88}\nОкно {wm} мес.  ·  пул {len(prof):,} займов  ·  "
          f"оценка РБ = {RB_ESTIMATE_KZT / 1e9:.0f} млрд ₸\n{'=' * 88}")
    print(table.to_string())
    hit = table.index[(table["H1_млрд"] * 1e9 >= RB_ESTIMATE_KZT)]
    print(f"\nH1 достигает 12 млрд ₸ при n ≥ {hit.min()}" if len(hit)
          else "\nH1 не достигает 12 млрд ₸ ни при одном протестированном n — "
               "оценка РБ не воспроизводится этим правилом.")
    print()

In [ ]:
# Разбивка по числу сорванных месяцев + историческая ставка срыва рядом с каждой
# строкой: сколько займов правило отпустило бы и сколько из них вернулось бы.
def phase_d_detail(n: int, window_months: int = LOOKBACK_MONTHS, K: int = 6) -> pd.DataFrame:
    prof = build_candidates(CANDIDATE_ASOF, CANDIDATE_LABEL, window_months)
    sets_ = rule_sets(prof, n)
    out = []
    for name, sub in sets_.items():
        for k, grp in sub.groupby("delinquent_months"):
            out.append({"правило": name, "сорванных_месяцев": int(k), **money_row(grp)})
        out.append({"правило": name, "сорванных_месяцев": -1, **money_row(sub)})  # -1 = итог
    df = pd.DataFrame(out)
    df["сорванных_месяцев"] = df["сорванных_месяцев"].replace(-1, "ИТОГО")
    return df.set_index(["правило", "сорванных_месяцев"])


N_FOR_DETAIL = 5     # допуск из постановки Дамира; поменяйте, когда n* зафиксирован
detail = phase_d_detail(N_FOR_DETAIL)
print(f"Разбивка при n = {N_FOR_DETAIL}, окно {LOOKBACK_MONTHS} мес.:")
print(detail.round(3).to_string())

hist = matrices[6]["pooled"]
near = min(THRESHOLDS, key=lambda t: abs(t - N_FOR_DETAIL))
rate = hist.loc[near, "rate"]
h1 = rule_sets(build_candidates(CANDIDATE_ASOF, CANDIDATE_LABEL, LOOKBACK_MONTHS),
               N_FOR_DETAIL)["H1"]
print(
    f"\nИсторическая ставка срыва при n={near}, K=6: {rate * 100:.1f}% "
    f"(база {int(hist.loc[near, 'denominator']):,})."
    f"\nПрименительно к H1 сегодня: из {len(h1):,} займов на "
    f"{h1['balance'].sum() / 1e9:.2f} млрд ₸ вернулось бы в 3-ю стадию порядка "
    f"{rate * len(h1):,.0f} займов на ~{rate * h1['balance'].sum() / 1e9:.2f} млрд ₸."
    f"\nЭто перенос исторической доли на сегодняшний состав, а не прогноз: срыв "
    f"\nкоррелирует с суммой, и разложение по остатку здесь не проверялось."
)

## Решение на одной картинке — сколько освобождаем против того, сколько вернётся

Phase C отвечала «насколько порог безопасен», Phase D — «сколько это в деньгах». По отдельности ни то, ни другое решения не принимает. Здесь обе оси на одном поле: по горизонтали — остаток, который правило H1 отпускает сегодня, по вертикали — доля срыва, показанная историей при том же `n`.

**Логарифм по горизонтали не для красоты.** Освобождаемый остаток идёт от 0.02 до 10.6 млрд, три порядка; на линейной шкале всё, что ниже `n=7`, схлопывается в ноль — а это ровно та половина картинки, вокруг которой спор.

**Вертикальные штрихи — вилка цензурирования**, не доверительный интервал: нижняя точка считает исчезнувшие без записи займы выжившими, верхняя — недосмотренными.

**H2 отмечен вертикалью, а не точкой, и это принципиально.** У H2 нет порога `n` — это проверка формы, а не уровня, — и historical re-default для его популяции не считался. Ставить его точкой означало бы приписать ему чужую ставку срыва.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Пороги, которые есть И в сетке Phase C, И в сетке кандидатов Phase D.
SHARED_NS = [n for n in CANDIDATE_THRESHOLDS if n in THRESHOLDS]
TRADEOFF_WINDOW = LOOKBACK_MONTHS

_prof = build_candidates(CANDIDATE_ASOF, CANDIDATE_LABEL, TRADEOFF_WINDOW)
h1_bn = {n: rule_sets(_prof, n)["H1"]["balance"].sum() / 1e9 for n in SHARED_NS}
h1_loans = {n: len(rule_sets(_prof, n)["H1"]) for n in SHARED_NS}
_h2 = rule_sets(_prof, SHARED_NS[0])["H2"]          # H2 не зависит от n
h2_bn, h2_loans = _h2["balance"].sum() / 1e9, len(_h2)

fig, ax = plt.subplots(figsize=(9.6, 6.2))
ax.set_yticks(range(0, 18, 2))
ax.grid(True, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_linewidth(0.8)

ax.axvline(RB_ESTIMATE_KZT / 1e9, color=INK_2, linewidth=1.2, linestyle=(0, (5, 3)), zorder=2)
ax.axvline(h2_bn, color=MUTED, linewidth=1.0, zorder=1)

y_top = max(b[1] for K in HORIZONS if bounds.get(K) is not None
            for b in zip(bounds[K]["нижняя_%"], bounds[K]["верхняя_%"])) * 1.30
ax.annotate(f"оценка РБ — {RB_ESTIMATE_KZT / 1e9:.0f} млрд тг",
            xy=(RB_ESTIMATE_KZT / 1e9, y_top), xytext=(-5, 0), textcoords="offset points",
            color=INK_2, ha="right", va="center", fontweight="bold", fontsize=8.5)
ax.annotate(f"H2 — {h2_bn:.2f} млрд", xy=(h2_bn, y_top * 0.92), xytext=(-5, 0),
            textcoords="offset points", color=MUTED, ha="right", va="center", fontsize=8.5)

for K in HORIZONS:
    if not matrices[K]["cohorts"]:
        continue
    b = bounds[K]
    xs = [h1_bn[n] for n in SHARED_NS]
    lo = [b.loc[n, "нижняя_%"] for n in SHARED_NS]
    hi = [b.loc[n, "верхняя_%"] for n in SHARED_NS]
    color = K_COLOR[K]
    for x, a, c in zip(xs, lo, hi):
        ax.plot([x, x], [a, c], color=color, linewidth=2.4, alpha=0.55,
                solid_capstyle="round", zorder=3)
    ax.plot(xs, lo, color=color, linewidth=1.4, alpha=0.45, zorder=3)
    ax.scatter(xs, lo, s=64, color=color, edgecolor=SURFACE, linewidth=2, zorder=5)
    # Метка на каждой точке: зелёный не проходит контраст к фону, и подписи — то самое
    # послабление, которого требует валидатор. Заодно несут n, третье измерение.
    for x, y, n in zip(xs, lo, SHARED_NS):
        ax.annotate(f"n={n}", xy=(x, y), xytext=(0, -14), textcoords="offset points",
                    color=INK_2, fontsize=7.5, ha="center", zorder=6)
    ax.annotate(f"K={K} мес.", xy=(xs[-1], lo[-1]), xytext=(10, 4),
                textcoords="offset points", color=color, fontweight="bold", fontsize=9.5)

ax.set_xscale("log")
ax.set_xlabel(f"освобождаемый остаток, млрд тг  ·  правило H1, окно {TRADEOFF_WINDOW} мес."
              f"  (логарифмическая шкала)")
ax.set_ylabel("историческая доля срыва, %")
ax.set_title("Что освобождаем против того, что вернётся — по каждому порогу n",
             loc="left", fontweight="bold")
fig.legend(handles=[
    Line2D([], [], marker="o", color=SURFACE, markerfacecolor=INK_2,
           markeredgecolor=SURFACE, markersize=8, lw=0, label="ставка (нижняя граница)"),
    Line2D([], [], color=INK_2, lw=2.4, alpha=0.55,
           label="вилка до верхней границы (цензурирование)"),
], loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

print("Те же числа таблицей — и как послабление к WARN по контрасту, и потому что "
      "с графика\nне снять точное значение:")
tbl = pd.DataFrame({
    "займов": pd.Series(h1_loans),
    "млрд_тг": pd.Series(h1_bn).round(2),
    **{f"K={K}_ставка_%": pd.Series(
        {n: f"{bounds[K].loc[n, 'нижняя_%']:.2f}–{bounds[K].loc[n, 'верхняя_%']:.2f}"
         for n in SHARED_NS}) for K in HORIZONS if matrices[K]["cohorts"]},
})
tbl.index.name = "n"
print(tbl.to_string())
print(f"\nH2 (не зависит от n): {h2_loans:,} займов, {h2_bn:.2f} млрд тг — "
      f"ставка срыва для этой популяции не считалась.")